In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install --upgrade pip
!pip install mediapipe-model-maker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.1.2
    Uninstalling pip-23.1.2:
      Successfully uninstalled pip-23.1.2
  Obtaining dependency information for mediapipe-model-maker from https://files.pythonhosted.org/packages/9a/f4/26b03c5b5cacd9bcbaf06a906915f56d6b7cd7276c28c246db4a42c1f35b/mediapipe_model_maker-0.2.1.3-py3-none-any.whl.metadata
  Obtaining dependency information for mediapipe>=0.10.0 from https://files.pythonhosted.org/packages/4a/58/1ec1516ee0f3e9f8073bf72a4c1653712b737068584a8cc91d1906d13a10/mediapipe-0.10.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for tf-models-official>=2.13.1 from https://files.pythonhosted.org/packages/d8/c3/4511e49412492966f4e6e0a20b41ae66d027e5a76e8c2b436b591ecc4273/tf_models_official-2.13.2-py2.py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.

In [3]:
import os
import tensorflow as tf
import mediapipe as mp
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt
from tqdm import tqdm

/opt/conda/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [4]:
dataset_path = "/kaggle/input/asl-alphabet-x-mediapipe"

#### Run a test on each image, if not hand landmarks are found, move it to the none folder (run only if original dataset)

In [5]:
"""
!cp -r {dataset_path} .

dataset_path = "asl_alphabet_train"
os.mkdir(dataset_path + "/none")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

count = 0

# for static images:
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5)

for folder in tqdm(os.listdir("asl_alphabet_train")):
    for file in os.listdir(f"asl_alphabet_train/{folder}"):
        image = cv2.imread(f"asl_alphabet_train/{folder}/{file}")
        
        # convert the BGR image to RGB before processing.
        results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

        if not results.multi_hand_landmarks:
            count = count + 1
            os.rename(f"asl_alphabet_train/{folder}/{file}", f"asl_alphabet_train/none/{file}")
            
print(f"Number of images without hands: {count}")
"""

'\n!cp -r {dataset_path} .\n\ndataset_path = "asl_alphabet_train"\nos.mkdir(dataset_path + "/none")\n\nmp_drawing = mp.solutions.drawing_utils\nmp_hands = mp.solutions.hands\n\ncount = 0\n\n# for static images:\nhands = mp_hands.Hands(\n    static_image_mode=True,\n    max_num_hands=2,\n    min_detection_confidence=0.5)\n\nfor folder in tqdm(os.listdir("asl_alphabet_train")):\n    for file in os.listdir(f"asl_alphabet_train/{folder}"):\n        image = cv2.imread(f"asl_alphabet_train/{folder}/{file}")\n        \n        # convert the BGR image to RGB before processing.\n        results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))\n\n        if not results.multi_hand_landmarks:\n            count = count + 1\n            os.rename(f"asl_alphabet_train/{folder}/{file}", f"asl_alphabet_train/none/{file}")\n            \nprint(f"Number of images without hands: {count}")\n'

In [6]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
    
print(sorted(labels))

/kaggle/input/asl-alphabet-x-mediapipe
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'none', 'space']


In [7]:
NUM_EXAMPLES = 3

for label in labels[:5]:
  label_dir = os.path.join(dataset_path, label)
  example_filenames = os.listdir(label_dir)[:NUM_EXAMPLES]
  fig, axs = plt.subplots(1, NUM_EXAMPLES, figsize=(10,2))
  for i in range(NUM_EXAMPLES):
    axs[i].imshow(plt.imread(os.path.join(label_dir, example_filenames[i])))
    axs[i].get_xaxis().set_visible(False)
    axs[i].get_yaxis().set_visible(False)
  fig.suptitle(f'Showing {NUM_EXAMPLES} examples for {label}')

plt.show()

In [8]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [9]:
hparams = gesture_recognizer.HParams(export_dir="exported_model")
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 29)                3741      
 out (Dense)                                                     
                                                             

In [10]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

5971/5971 [==============================] - 21s 2ms/step - loss: 0.2339 - categorical_accuracy: 0.9025
Test loss:0.2338545173406601, Test accuracy:0.9025288820266724


In [11]:
model.export_model()

Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite


In [12]:
!zip -r google_model.zip /kaggle/working/exported_model

  adding: kaggle/working/exported_model/ (stored 0%)
  adding: kaggle/working/exported_model/best_model_weights.index (deflated 56%)
  adding: kaggle/working/exported_model/best_model_weights.data-00000-of-00001 (deflated 9%)
  adding: kaggle/working/exported_model/gesture_recognizer.task (deflated 24%)
  adding: kaggle/working/exported_model/logs/ (stored 0%)
  adding: kaggle/working/exported_model/logs/train/ (stored 0%)
  adding: kaggle/working/exported_model/logs/train/events.out.tfevents.1696329550.1536af5c07ac.20.0.v2 (deflated 85%)
  adding: kaggle/working/exported_model/logs/validation/ (stored 0%)
  adding: kaggle/working/exported_model/logs/validation/events.out.tfevents.1696329611.1536af5c07ac.20.1.v2 (deflated 73%)
  adding: kaggle/working/exported_model/checkpoint (deflated 43%)
  adding: kaggle/working/exported_model/epoch_models/ (stored 0%)
  adding: kaggle/working/exported_model/epoch_models/model-0001.index (deflated 56%)
  adding: kaggle/working/exported_model/epoch_